In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!mamba install scikit-learn pandas matplotlib


mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, scikit-learn, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 3.349 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ brotli-python       1.2.0       py313h8faef1e_3      emscripten-forge-4x
+ ca-certificates     2026.10.23  hbd8a1cb_0           conda-forge
+ certifi             2026.10.23  pyhd8ed1ab_3         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.28.0      pyh5ded981_0         conda-forge
+ pandas              3.1.0       np23py313h124dbc7_0  emscripten-forge-4x
+ pysocks             1.7.1       py313h1804a44_3      emscripten-forge-4x
+ request

In [3]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
!mamba install scikit-learn xgboost pandas


mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, scikit-learn, pandas, xgboost
Channels: emscripten-forge-4x, conda-forge

Solving environment...


Cannot solve the request because of: xgboost * cannot be installed because there are no viable options:
├─ xgboost 3.4.2 would require
│  └─ py-xgboost ==3.4.2 cpu_pyh718b53a_0, which cannot be installed because there are no viable options:
│     └─ py-xgboost 3.4.2 would require
│        └─ libxgboost * cpu_h*_0, for which no candidates were found.
├─ xgboost 3.4.1 would require
│  └─ py-xgboost ==3.4.1 cpu_pyh35227d3_0, which cannot be installed because there are no viable options:
│     └─ py-xgboost 3.4.1 would require
│        └─ libxgboost * cpu_h*_0, for which no candidates were found.
├─ xgboost 3.4.0 would require
│  └─ py-xgboost ==3.4.0 cpu_pyhf9c7e24_0, which cannot be installed because there are no viable options:
│     └─ py-xgboost 3.4.0 would require
│        └─ libxgboost * cpu_h*_0, for which no candidates were found.
├─ xgboost 3.3.1 would require
│  └─ py-xgboost ==3.3.1 cpu_pyhf9c7e24_0, which cannot be installed because there are no viable options:
│     └─ py-xgb

In [4]:
df = pd.read_csv("../Data/Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df.drop(columns=["customerID"], inplace=True)


df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [5]:
# Tenure groups
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["New", "Short", "Medium", "Long"]
)

# Average monthly charge per tenure month
df["ChargePerTenure"] = (
    df["TotalCharges"] / (df["tenure"] + 1)
)

# High monthly charge indicator
df["HighMonthlyCharge"] = (
    df["MonthlyCharges"] > df["MonthlyCharges"].median()
).astype(int)

# Long-term contract indicator
df["LongTermContract"] = (
    df["Contract"] != "Month-to-month"
).astype(int)

print("Feature engineering completed!")
df.head()


Feature engineering completed!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,ChargePerTenure,HighMonthlyCharge,LongTermContract
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,0,New,14.925000,0,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,0,Medium,53.985714,0,1
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,1,New,36.050000,0,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,0,Medium,40.016304,0,1
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,1,New,50.550000,1,0


In [6]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include="object").columns.tolist()

# Include the tenure category
cat_cols += ["TenureGroup"]

print("Total features:", X.shape[1])
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))


Total features: 23
Numerical features: 7
Categorical features: 16


In [7]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=67,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Preprocessing completed!")


Training shape: (5282, 23)
Testing shape: (1761, 23)
Preprocessing completed!


In [12]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=15
    
)

X_train_selected = selector.fit_transform(
    X_train_processed, y_train
)

X_test_selected = selector.transform(X_test_processed)

print("Original features:", X_train_processed.shape[1])
print("Selected features:", X_train_selected.shape[1])
print("Feature selection completed!")


Original features: 52
Selected features: 15
Feature selection completed!


In [9]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)

print("Original features:", X_train_processed.shape[1])
print("PCA components:", X_train_pca.shape[1])
print("Explained variance:", round(sum(pca.explained_variance_ratio_), 4))


Original features: 52
PCA components: 20
Explained variance: 0.9547


In [13]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

ensemble_models = {
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6),
        n_estimators=50,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
}

trained_ensembles = {}

for name, model in ensemble_models.items():

    model.fit(X_train_selected, y_train)

    trained_ensembles[name] = model

    print(name, "trained successfully!")
    

Bagging trained successfully!
Random Forest trained successfully!


In [14]:
ensemble_results = []

for name, model in trained_ensembles.items():

    y_pred = model.predict(X_test_selected)
    y_prob = model.predict_proba(X_test_selected)[:, 1]

    ensemble_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

ensemble_df = pd.DataFrame(ensemble_results)

ensemble_df.round(4)


,Model,Accuracy,Precision,Recall,ROC-AUC
0,Bagging,0.7978,0.6442,0.5310,0.8332
1,Random Forest,0.7956,0.6361,0.5353,0.8262


In [15]:
print("Original features:", X_train_processed.shape[1])
print("PCA components:", X_train_pca.shape[1])
print("Variance retained:", round(sum(pca.explained_variance_ratio_), 4))


Original features: 52
PCA components: 20
Variance retained: 0.9547


In [16]:
pca_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=6),
    n_estimators=50,
    random_state=42
)

pca_bagging.fit(X_train_pca, y_train)

y_pred_pca = pca_bagging.predict(X_test_pca)
y_prob_pca = pca_bagging.predict_proba(X_test_pca)[:, 1]

print("PCA Bagging Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_pca), 4))
print("Precision:", round(precision_score(y_test, y_pred_pca), 4))
print("Recall:", round(recall_score(y_test, y_pred_pca), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_pca), 4))


PCA Bagging Results
Accuracy: 0.7967
Precision: 0.6461
Recall: 0.5161
ROC-AUC: 0.8281


In [17]:
final_model = trained_ensembles["Bagging"]

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", final_model)
])

joblib.dump(final_pipeline, "../models/final_churn_model.joblib")

print("Final model saved successfully!")


Final model saved successfully!


In [18]:
import os

print(os.path.exists("../models/final_churn_model.joblib"))


True


In [19]:
loaded_model = joblib.load("../models/final_churn_model.joblib")

predictions = loaded_model.predict(X_test.head(5))

print("Predictions:", predictions)


Predictions: [0 1 1 0 0]
